#### Convert PDF to Image (PNG)

In [ ]:
#!pip install pdf2image
# sudo apt update
# sudo apt install -y poppler-utils

In [ ]:
import os
import random
from pdf2image import convert_from_path
import re
from glob import glob
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader


In [ ]:
# Define paths
input_root = 'Training'
output_root = 'Training_images'

# Traverse the "training" directory
for subdir in ['Paper', 'Tablet']:
    input_subdir = os.path.join(input_root, subdir)
    output_subdir = os.path.join(output_root, subdir)

    # Ensure output subdirectory exists
    os.makedirs(output_subdir, exist_ok=True)

    # Loop through all PDF files in the current subdir
    for filename in os.listdir(input_subdir):
        if filename.lower().endswith('.pdf'):
            input_pdf_path = os.path.join(input_subdir, filename)
            basename = os.path.splitext(filename)[0]

            # Convert PDF to images (one per page)
            try:
                images = convert_from_path(input_pdf_path)
                for i, img in enumerate(images):
                    img_filename = f"{basename}_page_{i+1}.png"
                    img_path = os.path.join(output_subdir, img_filename)
                    img.save(img_path, 'PNG')
                print(f"✅ Converted: {input_pdf_path}")
            except Exception as e:
                print(f"❌ Failed to convert {input_pdf_path}: {e}")

#### Get Lists of Paper and Tablet files

In [ ]:
paper_groups, tab_groups = None, None

for directory in os.listdir("Training_images"):
    print(directory)
    if directory == "Paper":
        images = os.listdir("Training_images/"+directory)
        images = ["Training_images/"+directory+'/'+x for x in images]
        images.sort()
        paper_groups= images
    else:
        images = os.listdir("Training_images/"+directory)
        images = ["Training_images/"+directory+'/'+x for x in images]
        images.sort()
        tab_groups = images

In [ ]:
len(paper_groups), len(tab_groups)

#### Train Test Split

In [ ]:
# Train Test Split (90, 10)
# 80% training, 10% validation, 10% testing


paper_groups_train, paper_groups_test = paper_groups[:151], paper_groups[151:]
tab_groups_train, tab_groups_test = tab_groups[:71], tab_groups[71:]

full_train_data = paper_groups_train + tab_groups_train
full_test_data = paper_groups_test + tab_groups_test

len(full_train_data), len(full_test_data)

#### Create Pairs (Same Writer and Different Writers)

In [ ]:
def build_writer_dict_from_list(image_paths):
    """
    Build a dictionary mapping writer IDs to lists of image paths,
    based on a list of image file paths.

    Args:
        image_paths (list): List of full image file paths (e.g., .png).

    Returns:
        dict: {writer_id: [list of image paths]}
    """
    writer_dict = {}
    id_pattern = re.compile(r'(id_[A-Za-z0-9]+)')

    for img_path in image_paths:
        match = id_pattern.search(os.path.basename(img_path))
        if match:
            writer_id = match.group(1)
            writer_dict.setdefault(writer_id, []).append(img_path)

    return writer_dict


writer_dict = build_writer_dict_from_list(full_train_data)
writer_dict

In [ ]:
def generate_pairs(num_pairs, writer_dict, writer_ids):
    pairs = []

    for _ in range(num_pairs):
        if random.random() < 0.5:
            # Positive pair (same writer)
            writer = random.choice(writer_ids)
            if len(writer_dict[writer]) >= 2:
                img1, img2 = random.sample(writer_dict[writer], 2)
                label = 1
            else:
                continue  # skip this one
            
        else:
            # Negative pair (different writers)
            w1, w2 = random.sample(writer_ids, 2)
            img1 = random.choice(writer_dict[w1])
            img2 = random.choice(writer_dict[w2])
            label = 0
            
        pairs.append((img1, img2, label))
    return pairs

In [ ]:
pairs = generate_pairs(num_pairs=500, writer_dict=writer_dict, writer_ids=list(writer_dict.keys()))